In [21]:
import pandas as pd

In [19]:
df2 = pd.read_csv("data2.csv")
df2

,uid,dt,amount,transaction
0,1,2023-10-01 07:09:36.011140,14000,deposit
1,1,2023-10-01 10:09:36.011140,12000,deposit
2,1,2023-10-01 12:09:36.011140,11000,deposit
3,1,2023-10-01 11:09:36.011140,1000,withdraw
4,1,2023-10-01 15:09:36.011140,4000,deposit
5,1,2023-10-01 11:09:36.011140,4000,withdraw
6,2,2023-10-02 01:09:36.011140,51000,deposit
7,2,2023-10-03 02:09:36.011140,1000,withdraw
8,4,2023-10-03 11:09:36.011140,20000,deposit
9,2,2023-10-04 12:09:36.011140,1000,withdraw


What’s the amount of the latest deposit of users who registered on 1 and 2 Oct 2023.
Output: uid, amount, dt
 
 
What’s the sum of amount of the 2 latest deposit of users who registered on 1 and 2 Oct 2023.
Output: uid, amount
 

In [2]:
import pandas as pd

df = pd.read_csv("data.csv")
df2 = pd.read_csv("data2.csv")

# แปลง datetime
df['dt'] = pd.to_datetime(df['dt'])
df2['dt'] = pd.to_datetime(df2['dt'])

# # หา user_id ที่สมัครใน 1-2 Oct 2023
registered_users = df[(df['account_action'] == 'register') & 
                      (df['dt'].dt.date.isin([pd.Timestamp('2023-10-01').date(), 
                                               pd.Timestamp('2023-10-02').date()]))]

# หา unique ids
registered_uids = registered_users['uid'].unique()

# Filter df2 เฉพาะ deposits จาก registered users
deposits = df2[(df2['uid'].isin(registered_uids)) & 
               (df2['transaction'] == 'deposit')]

# ดึง deposit ล่าสุด (วันสุดท้าย) ของแต่ละ users
result = deposits.sort_values('dt').groupby('uid').tail(1)[['uid', 'amount', 'dt']]
print(result)

    uid  amount                         dt
4     1    4000 2023-10-01 15:09:36.011140
16    2   21000 2023-10-07 21:09:36.011140


In [18]:
df = pd.read_csv("data.csv")
df2 = pd.read_csv("data2.csv")
df['dt'] = pd.to_datetime(df['dt'])
df2['dt'] = pd.to_datetime(df2['dt'])

# registration dates - วันที่ 1,2 Oct 2023
target_dates = {pd.Timestamp('2023-10-01').date(), pd.Timestamp('2023-10-02').date()}

# หา user_id ที่สมัคร
registered = df[(df['account_action'] == 'register') & df['dt'].notna()]
registered = registered[registered['dt'].dt.date.isin(target_dates)]
registered_uids = registered['uid'].unique()

# หา transactions ที่เป็น deposit ของ user
deposits = df2[(df2['uid'].isin(registered_uids)) & (df2['transaction'] == 'deposit')].copy()

# sort uid กับ datetime จากมากไปน้อย
deposits.sort_values(['uid', 'dt'], ascending=[True, False], inplace=True)

# เอาเฉพาะ 2 แถวล่าสุดต่อ uid
top2 = deposits.groupby('uid', as_index=False).head(2)

# sum amounts ของแต่ละ uid
sums = top2.groupby('uid', as_index=False)['amount'].sum()
sums = sums.rename(columns={'amount': 'amount'})

print(sums)

   uid  amount
0    1   15000
1    2   26000
